# Sri Lanka Air Quality: Quickstart

A first look at the **Sri Lanka DSD-Level Air Quality** dataset: daily air pollution and US AQI for all 339 Divisional Secretariat Divisions (DSDs), from 2023 to yesterday.

In this notebook:
1. Load the data
2. Check what's in it
3. Find the most polluted districts
4. See how often the air is good or bad
5. Look at seasonal patterns
6. Follow one area over time
7. Map air quality across the country

> Values are CAMS model estimates (via Open-Meteo) on a ~45 km grid, not station readings.  
> Source code and docs: [github.com/th33k/dataset-sri-lanka-air-quality](https://github.com/th33k/dataset-sri-lanka-air-quality)

## 1. Load the data
The dataset has one CSV per year. This cell finds them in `/kaggle/input` (on Kaggle) or downloads them with `kagglehub` (anywhere else).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def find_data_dir():
    hits = sorted(Path("/kaggle/input").glob("**/sri_lanka_dsd_air_quality_*.csv"))
    if hits:
        return hits[0].parent
    import kagglehub
    return Path(kagglehub.dataset_download("theekshanaudara/sri-lanka-air-quality"))

DATA_DIR = find_data_dir()
files = sorted(DATA_DIR.glob("sri_lanka_dsd_air_quality_*.csv"))
df = pd.concat((pd.read_csv(f, parse_dates=["date"]) for f in files), ignore_index=True)

print(f"Loaded {len(files)} files, {len(df):,} rows")
print(f"Dates: {df['date'].min():%Y-%m-%d} to {df['date'].max():%Y-%m-%d}")
print(f"Areas: {df['dsd_name'].nunique()} DSDs, {df['district_name'].nunique()} districts, {df['province_name'].nunique()} provinces")

## 2. What's in it
Each row is one DSD on one day. Pollutants are in µg/m³; AQI is the US EPA scale (0–500).

In [ ]:
df.head()

In [ ]:
df.describe().T[["mean", "min", "max"]].round(2)

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))

## 3. Most polluted districts
Average daily PM2.5 per district. The WHO 24-hour guideline is 15 µg/m³.

In [ ]:
district = (df.groupby("district_name")["pm2_5_mean"].mean().sort_values())

ax = district.plot.barh(figsize=(9, 8), color="#c0504d")
ax.axvline(15, ls="--", color="grey")
ax.text(15.2, 0, "WHO guideline", color="grey")
ax.set(title="Average PM2.5 by district", xlabel="PM2.5 (µg/m³)", ylabel="")
plt.tight_layout()

district.sort_values(ascending=False).head(5).round(2)

## 4. How often is the air good or bad?
`aqi_category` is based on the worst hour of each day.

In [ ]:
order = ["Good", "Moderate", "Unhealthy for Sensitive Groups",
         "Unhealthy", "Very Unhealthy", "Hazardous"]
colors = ["#00e400", "#ffff00", "#ff7e00", "#ff0000", "#8f3f97", "#7e0023"]

share = df["aqi_category"].value_counts(normalize=True).reindex(order).fillna(0) * 100
ax = share.plot.bar(color=colors, edgecolor="black")
ax.set(title="Share of DSD-days by AQI category", ylabel="% of days", xlabel="")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

share.round(2)

## 5. Seasonal pattern
Average AQI across the whole country, by month.

In [ ]:
monthly = df.set_index("date")["us_aqi_mean"].resample("MS").mean()

ax = monthly.plot(marker="o", color="#4472c4")
ax.set(title="Country-wide average AQI by month", ylabel="US AQI", xlabel="")
plt.tight_layout()

In [ ]:
by_month = df.groupby(df["date"].dt.month)["pm2_5_mean"].mean()
ax = by_month.plot.bar(color="#4472c4")
ax.set(title="Average PM2.5 by calendar month (all years)", xlabel="Month", ylabel="PM2.5 (µg/m³)")
plt.xticks(rotation=0)
plt.tight_layout()

## 6. One area over time
Change `AREA` to any DSD name (see `df['dsd_name'].unique()`).

In [ ]:
AREA = "Colombo"

area = df[df["dsd_name"] == AREA].set_index("date").sort_index()
ax = area["pm2_5_mean"].plot(alpha=0.35, label="Daily mean")
area["pm2_5_mean"].rolling(30).mean().plot(ax=ax, lw=2, label="30-day average")
ax.set(title=f"PM2.5 in {AREA} DSD", ylabel="PM2.5 (µg/m³)", xlabel="")
ax.legend()
plt.tight_layout()

area[["pm2_5_mean", "pm10_mean", "nitrogen_dioxide_mean", "us_aqi_max", "aqi_category"]].tail()

## 7. Map
Each dot is a DSD, coloured by its average AQI over the whole record.

In [ ]:
dsd = df.groupby(["dsd_name", "latitude", "longitude"], as_index=False)["us_aqi_mean"].mean()

fig, ax = plt.subplots(figsize=(6, 8))
sc = ax.scatter(dsd["longitude"], dsd["latitude"], c=dsd["us_aqi_mean"], cmap="RdYlGn_r", s=30)
fig.colorbar(sc, label="Average US AQI")
ax.set(title="Average AQI by DSD", xlabel="Longitude", ylabel="Latitude")
ax.set_aspect("equal")
plt.tight_layout()

## Next steps
- **Forecast** tomorrow's AQI with the lag features of each DSD
- **Compare** pollutants: which one drives the AQI in each region?
- **Join** with weather or population data using `dsd_pcode`

If you find this dataset useful, an upvote helps others find it.